In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_26.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['sample_discourse_id']
me:  2
trying: ['train_first_last']
me:  11
trying: ['get_n_grams_2']
me:  25
trying: ['warnings']
me:  0
trying: ['other_words_to_take_out']
me:  22
trying: ['counts_dict']
me:  26
trying: ['cols_to_merge']
me:  13
trying: ['test_names']
me:  26
trying: ['f']
me:  26
trying: ['train']
me:  22
trying: ['counter']
me:  11
trying: ['cols_to_display']
me:  14
trying: ['last_ones']
me:  14
trying: ['stop_english']
me:  22
trying: ['text']
me:  22
trying: ['data']
me:  12
trying: ['bigrams']
me:  23
trying: ['test_txt']
me:  1
trying: ['ax2']
me:  8
trying: ['train_last']
me:  11
trying: ['word_dict']
me:  12
trying: ['get_n_grams']
me:  23
trying: ['len_dict']
me:  12
trying: ['train_texts']
me:  26
trying: ['style']
me:  0
trying: ['tqdm']
me:  0
trying: ['av_per_essay']
me:  8
trying: ['all_gaps']
me:  20
trying: ['glob']
me:  0
trying: ['add_gap_rows']
me:  20
trying: ['stopword

In [4]:
%%RecordEvent
%%time
### cell 26 ###

# 1. Restrict to first 5 rows (GPU)
train_text_df = train_text_df.head(5)

# 1. Tokenize each essay into one row per token, with token indices
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=train_text_df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
tokenized['token_index'] = tokenized.groupby('id').cumcount()
print(1)

# 2. Expand the discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    # split into string tokens and explode (all on GPU)
    .assign(token_index_str=lambda df: df['predictionstring'].str.split(' '))
    .explode('token_index_str')
    .reset_index(drop=True)
)
# convert token indices to ints, compute position in discourse
labels['token_index'] = labels['token_index_str'].astype('int32')
labels['pos_in_discourse'] = labels.groupby('disc_row').cumcount()
# build B-/I- labels vectorized
labels['label'] = (
    ('B-' + labels['discourse_type'])
    .where(labels['pos_in_discourse'] == 0,
           'I-' + labels['discourse_type'])
)
# keep only needed columns
labels = labels[['id', 'token_index', 'label']]
print(2)

# 3. Merge tokenized text with labels, defaulting missing labels to "O"
tokenized = tokenized.merge(labels, on=['id', 'token_index'], how='left')
tokenized['label'] = tokenized['label'].fillna('O')
print(3)

# 4. Aggregate back to one list of labels per essay (GPU list aggregation)
entities_df = (
    tokenized.groupby('id')['label']
    .agg_list()
    .reset_index()
    .rename(columns={'label': 'entities'})
)
print(4)

# 5. Attach the entities column to the original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')
print(5)
print(6)

1
2
3


AttributeError: 'SeriesGroupBy' object has no attribute 'agg_list'

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_26/checkpoints/post_cell_26_try_8.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_26_try_8.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [ ]:
opt_output = Out.get(4)